In [0]:
dbutils.widgets.text("kaggle_username", "")
dbutils.widgets.text("kaggle_key", "")

In [0]:
dbutils.secrets.get("nba_insights", "kaggle_key")

In [0]:
pip install kagglehub[pandas-datasets]

In [0]:
import os
os.environ["KAGGLE_USERNAME"] = dbutils.secrets.get("nba_insights", "kaggle_username")
os.environ["KAGGLE_KEY"] = dbutils.secrets.get("nba_insights", "kaggle_key")

In [0]:
# Install dependencies as needed:

import kagglehub
from kagglehub import KaggleDatasetAdapter


def get_kaggle_data(file_path):
  return kagglehub.dataset_load(
    KaggleDatasetAdapter.PANDAS,
    "wyattowalsh/basketball",
    file_path,
    # Provide any additional arguments like 
    # sql_query or pandas_kwargs. See the 
    # documenation for more information:
    # https://github.com/Kaggle/kagglehub/blob/main/README.md#kaggledatasetadapterpandas
  )



In [0]:
catalog_name = "nba_insights"
schema_name = "bronze"
games_table = "games"
common_player_info_table = "common_player_info"
inactive_players_table = "inactive_players"

## Ingest game

In [0]:
from pyspark.sql.functions import current_timestamp

game_df = get_kaggle_data("csv/game.csv")

game_df = spark.createDataFrame(game_df)
game_df\
    .withColumn("ingestion_timestamp", current_timestamp())\
        .write\
        .mode("overwrite")\
            .saveAsTable(f"{catalog_name}.{schema_name}.{games_table}")




## Ingest Common Player Info

In [0]:
from pyspark.sql.functions import current_timestamp;

common_player_info_pd = get_kaggle_data("csv/common_player_info.csv")

print(common_player_info_pd.dtypes)
common_player_info_df = spark.createDataFrame(common_player_info_pd)
common_player_info_df\
    .withColumn("ingestion_timestamp", current_timestamp())\
        .write.mode("overwrite")\
                .saveAsTable(f"{catalog_name}.{schema_name}.{common_player_info_table}")




## Ingest Inactive Players

In [0]:
inactive_players_pd = get_kaggle_data("csv/inactive_players.csv")
inactive_player_df = spark.createDataFrame(inactive_players_pd)

inactive_player_df\
    .withColumn("ingestion_timestamp", current_timestamp())\
        .write.mode("overwrite").saveAsTable(f"{catalog_name}.{schema_name}.{inactive_players_table}")
